# **MOE架构概述与基础知识**

**导航**：[返回课程主页](02.01_chapter_intro.ipynb) | [下一章：MoE并行策略 →](02.03_parallel_strategy.ipynb)

---

本章将介绍Mixture of Experts (MoE) 架构的基础理论，内容涵盖基本概念、核心组件、激活函数及计算流程。

**本章目录**（点击跳转）：
- [什么是MoE架构](#1-什么是moe架构)
- [MoE的核心组件](#2-moe的核心组件)
- [门控网络（Gating Network）](#3-门控网络gating-network)
- [专家网络（Expert Network）](#4-专家网络expert-network)
- [Top-K路由策略](#5-top-k路由策略)
- [MoE计算流程](#6-moe计算流程)
- [MoE与MC2融合算子的关系](#7-moe与mc2融合算子的关系)
- [课后练习](#课后练习)

---
## **1. 什么是MoE架构**
MoE（Mixture of Experts，混合专家模型）是一种深度学习架构，通过**稀疏激活**机制，在保持模型规模增长的同时，显著降低实际计算量。MoE的核心思想是：对于每个输入样本，只激活网络中的一小部分参数（专家），而非全部参数参与计算。

### **MoE的核心优势**

1. **参数规模扩展**：模型可拥有万亿级参数，而每次前向传播只计算其中一小部分
2. **计算效率提升**：通过稀疏激活，实际FLOPs远小于总参数量对应的计算量
3. **专业化分工**：不同专家学习不同类型的特征，提升模型表达能力
4. **训练稳定性**：专家负载均衡机制确保训练过程稳定收敛

## **2. MoE的核心组件**
MoE层通常替换Transformer中的前馈网络（FFN）层，由两个核心组件构成：**门控网络（Gating Network）**和**专家网络（Expert Network）**。

### **MoE层结构**

<img src="./images/02.02_moe_layer_structure.png" width="300">

### **核心组件说明**



门控网络 : 决定Token应该由哪些专家处理。

专家网络 : 实际处理Token的计算单元。

## **3. 门控网络（Gating Network）**
门控网络负责计算每个Token与每个专家之间的相关性分数，并选择Top-K个专家来处理该Token。

### **门控网络计算流程**

```
输入: x ∈ R^{B×S×D}  (Batch × Sequence × Hidden_Dim)

Step 1: 计算路由分数
    scores = x @ W_g          # W_g ∈ R^{D×E}, E为专家数量
    # scores ∈ R^{B×S×E}

Step 2: Softmax归一化
    gate_scores = softmax(scores, dim=-1)
    # gate_scores ∈ R^{B×S×E}, 每个Token对所有专家的权重分布

Step 3: Top-K选择
    topk_scores, topk_indices = topk(gate_scores, k)
    # topk_scores ∈ R^{B×S×K}
    # topk_indices ∈ R^{B×S×K}, K为每个Token激活的专家数

Step 4: 归一化Top-K权重
    weights = topk_scores / sum(topk_scores)
    # 归一化确保权重和为1
```

### **门控网络参数**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">参数</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">说明</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">典型值</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>E</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家总数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">8, 16, 64, 128</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>K</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每Token激活专家数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">1, 2, 4</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>D</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">隐藏层维度</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">4096, 5120, 8192</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>W_g</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">门控权重矩阵</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">D×E</td>
  </tr>
</table>

---
## **4. 专家网络（Expert Network）**
专家网络是实际执行计算的组件。每个专家通常是一个前馈神经网络（FFN），结构与标准Transformer的FFN相同。

### **专家网络实现形式**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">形式</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">说明</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">参数量</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">独立专家</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每个专家独立的权重矩阵</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">E × (D×4D + 4D×D)</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">共享专家</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">部分专家共享权重</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">降低参数冗余</td>
  </tr>
</table>

### **MoE FFN计算流程**

```
对于每个Token t:
1. 门控网络选择Top-K专家: experts = [e_1, e_2, ..., e_K]
2. 获取对应权重: weights = [w_1, w_2, ..., w_K]
3. 每个选中的专家处理Token:
   output_i = Expert_e_i(input_t)
4. 加权聚合:
   final_output = Σ(w_i × output_i)
```
---
## **5. Top-K路由策略**
Top-K路由是MoE稀疏激活的关键机制，决定了每个Token激活哪些专家。

### **Top-K路由算法**

```python
def topk_routing(x, W_g, num_experts, top_k):
    """
    x: [batch, seq_len, hidden_dim]
    W_g: [hidden_dim, num_experts]
    """
    # 计算路由分数
    router_logits = x @ W_g  # [batch, seq_len, num_experts]
    
    # Softmax归一化
    router_probs = softmax(router_logits, dim=-1)
    
    # 选择Top-K专家
    topk_probs, topk_indices = topk(router_probs, k=top_k)
    
    # 重新归一化
    topk_probs = topk_probs / sum(topk_probs, dim=-1, keepdim=True)
    
    return topk_probs, topk_indices
```

### **负载均衡策略**

为确保训练稳定，需要引入负载均衡机制，避免专家负载不均：

#### **1. 辅助损失函数（Auxiliary Loss）**
```
L_aux = α × Σ(f_i × P_i)

其中:
- f_i: 专家i被选中的频率
- P_i: 专家i的平均路由概率
- α: 平衡系数，通常为0.01
```

#### **2. 专家容量（Expert Capacity）**
```
capacity = (tokens_per_batch / num_experts) × capacity_factor

- capacity_factor: 通常为1.25~2.0
- 超出容量的Token被丢弃或路由到溢出专家
```

#### **3. 噪声Top-K（Noisy Top-K）**
```
noise = randn() × softplus(noise_weight)
router_logits = router_logits + noise
```
引入噪声增加探索性，避免专家选择过于集中。




## **6. MOE计算流程**


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# MoE基础组件演示
class Expert(nn.Module):
    """单个专家网络（标准FFN）"""
    def __init__(self, hidden_dim, intermediate_dim):
        super().__init__()
        self.up_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.down_proj = nn.Linear(intermediate_dim, hidden_dim, bias=False)
        self.act = nn.GELU()
    
    def forward(self, x):
        return self.down_proj(self.act(self.up_proj(x)))


class MoELayer(nn.Module):
    """简化版MoE层实现"""
    def __init__(self, hidden_dim, intermediate_dim, num_experts, top_k):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_experts = num_experts
        self.top_k = top_k
        
        # 门控网络
        self.gate = nn.Linear(hidden_dim, num_experts, bias=False)
        
        # 专家网络
        self.experts = nn.ModuleList([
            Expert(hidden_dim, intermediate_dim) 
            for _ in range(num_experts)
        ])
    
    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        
        # Step 1: 门控计算
        gate_logits = self.gate(x)  # [B, S, E]
        gate_probs = F.softmax(gate_logits, dim=-1)
        
        # Step 2: Top-K选择
        topk_probs, topk_indices = torch.topk(gate_probs, self.top_k, dim=-1)
        # 归一化
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)
        
        # Step 3 & 4: 专家计算与聚合
        output = torch.zeros_like(x)
        
        for k in range(self.top_k):
            # 获取第k个专家的索引和权重
            expert_idx = topk_indices[:, :, k]  # [B, S]
            expert_weight = topk_probs[:, :, k:k+1]  # [B, S, 1]
            
            # 对每个专家处理
            for e in range(self.num_experts):
                # 找出分配给专家e的token
                mask = (expert_idx == e)  # [B, S]
                if not mask.any():
                    continue
                
                # 提取token并计算
                expert_input = x[mask]  # [num_tokens, D]
                expert_output = self.experts[e](expert_input)
                
                # 加权聚合
                weighted_output = expert_output * expert_weight[mask]
                output[mask] += weighted_output
        
        return output

# 演示

print("=== MoE层计算演示 ===")
batch_size, seq_len = 2, 8
hidden_dim, intermediate_dim = 64, 256
num_experts, top_k = 4, 2

moe = MoELayer(hidden_dim, intermediate_dim, num_experts, top_k)
x = torch.randn(batch_size, seq_len, hidden_dim)

print(f"输入形状: {x.shape}")
print(f"专家数量: {num_experts}")
print(f"每Token激活专家数: {top_k}")

# 查看门控输出

with torch.no_grad():
    gate_logits = moe.gate(x)
    gate_probs = F.softmax(gate_logits, dim=-1)
    topk_probs, topk_indices = torch.topk(gate_probs, moe.top_k, dim=-1)
    
print(f"\n门控概率分布形状: {gate_probs.shape}")
print(f"Top-K索引形状: {topk_indices.shape}")
print(f"\n第一个Token的专家分配:")
print(f"  选中的专家索引: {topk_indices[0, 0].tolist()}")
print(f"  对应权重: {topk_probs[0, 0].tolist()}")

# 前向传播

output = moe(x)
print(f"\n输出形状: {output.shape}")

## **7. MoE与MC2融合算子的关系**
MoE模型在分布式训练中面临显著的通信挑战，MC2融合算子可以有效解决这些问题。

### **MoE分布式训练的通信挑战**

<img src="./images/02.02_alltoall_communication.png" width="1000">

**说明**：发送端按目标专家分发Token，接收端按本地专家汇聚Token。

### **通信开销分析**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">通信阶段</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">通信类型</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">数据量</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">瓶颈</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Token分发</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">2×Token×K</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Token到专家映射</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家计算</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">无通信</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">-</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">本地计算</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">结果聚合</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">2×Token×K</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家到Token还原</td>
  </tr>
</table>

### **MC2融合算子优化MoE**

#### **AllToAll + Expert计算融合**
```
传统模式:
AllToAll(Token分发) → 等待 → Expert计算 → AllToAll(结果聚合)

MC2融合模式:
AllToAll(Token分发) + Expert计算(部分) 重叠执行
```

### **MC2优化效果**

- **隐藏通信延迟**：AllToAll通信与专家计算重叠
- **减少内存访问**：Token直接从通信缓冲区进入专家计算
- **提升资源利用**：计算单元与通信单元并行工作
---
## **8. 常见MoE变体**
### **1. Switch Transformer**
- 每Token只激活1个专家（K=1）
- 简化路由策略，降低计算开销
- 引入专家容量概念处理溢出

### **2. Expert Choice**
- 专家选择Token，而非Token选择专家
- 天然负载均衡
- 容量因子精确控制

### **3. DeepSeek-MoE**
- 细粒度专家分割
- 共享专家 + 路由专家混合架构
- 降低专家冗余，提升专业化

### **4. Mixtral 8x7B**
- 8个专家，每Token激活2个
- 总参数47B，激活参数13B
- 与稠密模型推理效率接近
---
# **课后练习**
请根据本章内容认真思考，选出正确选项。  

1. （单选题）MoE架构的核心优势是什么？  
    A. 通过稀疏激活，解耦参数量与计算量。    
    B. 减少模型总参数量。     
    C. 只使用单个专家进行计算。    
    D. 消除所有通信开销。  


2. （单选题）在MoE中，门控网络的作用是什么？  
    A. 存储所有专家的权重。  
    B. 直接计算Token的最终输出。   
    C. 训练所有专家网络。  
    D. 计算Token与专家的相关性，选择Top-K专家。    

3. （单选题）Top-K路由中K=2表示什么？  
    A. 模型有2个专家。  
    B. 每个Token激活2个专家。   
    C. 训练进行2轮迭代。  
    D. 模型有2层。    

4. （单选题）MoE分布式训练中，主要的通信开销来自？    
    A. 门控网络计算。  
    B. AllToAll通信进行Token分发和聚合。   
    C. 单个专家的FFN计算。  
    D. 激活函数计算。    

5. （多选题）关于MoE架构，以下说法正确的是？    
    A. 每个Token会激活所有专家。   
    B. 辅助损失函数用于实现负载均衡。   
    C. MC2融合算子可以优化MoE的AllToAll通信。  
    D. 专家网络通常是FFN结构。    

---

# **查看答案**

In [ ]:
!cat answer/02.02_answer.txt

---
**导航**：[返回课程主页](02.01_chapter_intro.ipynb) | [下一章：MoE并行策略 →](02.03_parallel_strategy.ipynb)

---